In [ ]:
import re
import os
import requests
import html2text
import time
from bs4 import BeautifulSoup
from bs4 import Comment
from collections import deque

pattern = r"https?:\/\/(?:www\d*\.)?eecs\.berkeley\.edu(?:\/[^\s]*)?"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

os.makedirs("html", exist_ok=True)
os.makedirs("cleaned_text", exist_ok=True)

h2t = html2text.HTML2Text()
h2t.ignore_links = True
h2t.ignore_images = True
h2t.ignore_tables = False
h2t.body_width = 0


def normalize_url(url):
    """Strip fragment and trailing slash for deduplication."""
    url = url.split("#")[0].rstrip("/")
    return url


def url_to_safe_name(url):
    url = normalize_url(url)
    url = url.replace("http://", "").replace("https://", "")
    return re.sub(r"[^\w-]", "_", url)


def get_matching_urls(html, base_url=None):
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception:
        return []

    urls = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if href.startswith("/") and base_url:
            href = base_url.rstrip("/") + href

        href = normalize_url(href)

        if re.match(pattern, href):
            urls.append(href)

    return urls

In [ ]:
seed = "https://eecs.berkeley.edu"
max_pages = 100000
retry_delay = 5   # seconds to wait on first 429; doubles each consecutive hit

# --- Restart: rebuild visited + queue from already-saved HTML files ---
visited = set()
queue = deque()

print("Scanning existing html/ for already-crawled pages...")
for filename in os.listdir("html"):
    if not filename.endswith(".html"):
        continue
    safe_name = filename[:-5]
    visited.add(safe_name)

# Re-read saved HTML to reconstruct the link frontier.
for filename in os.listdir("html"):
    if not filename.endswith(".html"):
        continue
    with open(f"html/{filename}") as f:
        html = f.read()
    safe_name = filename[:-5]
    for u in get_matching_urls(html):
        if url_to_safe_name(u) not in visited:
            queue.append(u)

# If nothing was found (fresh start), begin from seed.
if not visited and not queue:
    queue.append(seed)

print(f"Resuming: {len(visited)} pages already crawled, {len(queue)} URLs queued.\n")
# ---------------------------------------------------------------------

backoff = retry_delay  # tracks current wait time for 429s

while queue and len(visited) < max_pages:
    url = queue.popleft()
    safe_name = url_to_safe_name(url)

    if safe_name in visited:
        continue

    print(f"Visiting ({len(visited)}/{max_pages}): {url}")

    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
    except Exception as e:
        print(f"  Request error: {e} — skipping.")
        visited.add(safe_name)
        continue

    # 429 Too Many Requests — back off and retry
    if response.status_code == 429:
        print(f"  429 Too Many Requests — waiting {backoff}s before retrying...")
        time.sleep(backoff)
        backoff = min(backoff * 2, 120)  # exponential backoff, cap at 2 min
        queue.appendleft(url)            # put it back at the front
        continue

    # Reset backoff after a successful request
    backoff = retry_delay

    # 404 or other non-2xx — skip permanently
    if response.status_code == 404:
        print(f"  404 Not Found — skipping.")
        visited.add(safe_name)
        continue

    if not response.ok:
        print(f"  HTTP {response.status_code} — skipping.")
        visited.add(safe_name)
        continue

    # Cloudflare challenge page
    if "Just a moment..." in response.text:
        print(f"  Cloudflare challenge — waiting {backoff}s before retrying...")
        time.sleep(backoff)
        backoff = min(backoff * 2, 120)
        queue.appendleft(url)
        continue

    if "Forbidden" in response.text:
        visited.add(safe_name)
        continue

    visited.add(safe_name)
    with open(f"html/{safe_name}.html", "w") as f:
        f.write(response.text)

    new_urls = get_matching_urls(response.text, base_url=url)
    for u in new_urls:
        if url_to_safe_name(u) not in visited:
            queue.append(u)

print(f"\nDone. Crawled {len(visited)} pages.")

In [ ]:
def clean_html_for_rag(html):
    """Extract mostly-main textual content from HTML for RAG.

    Goals:
    - Prefer the page's main/article content when available.
    - Remove boilerplate (nav/headers/footers/sidebars/cookie banners).
    - Preserve basic structure (headings/lists/tables) via html2text.
    - Normalize whitespace for easier chunking/embedding.
    """
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception:
        return ""

    # Ignore html comments
    try:
        for c in soup.find_all(string=lambda t: isinstance(t, Comment)):
            c.extract()
    except Exception:
        pass

    # Remove rendering-only tags
    for tag in soup(["script",
        "style",
        "noscript",
        "svg",
        "img",
        "picture",
        "source",
        "canvas",
        "iframe",
        "object",
        "embed",
        "form",
        "input",
        "button",
        "select",
        "textarea",
        "link",
        "meta",
    ]):
        tag.decompose()

    # Remove hidden / aria-hidden content
    for tag in soup.find_all(True):
        try:
            if tag.has_attr("hidden"):
                tag.decompose()
                continue
            if str(tag.attrs.get("aria-hidden", "")).lower() == "true":
                tag.decompose()
                continue
            style = str(tag.attrs.get("style", "")).lower()
            if "display:none" in style or "visibility:hidden" in style:
                tag.decompose()
                continue
        except Exception:
            continue

    noise_classes = {
        "nav",
        "navbar",
        "menu",
        "breadcrumb",
        "breadcrumbs",
        "footer",
        "header",
        "masthead",
        "sidebar",
        "aside",
        "cookie",
        "consent",
        "banner",
        "popup",
        "modal",
        "subscribe",
        "newsletter",
        "social",
        "share",
        "related",
        "promo",
        "advert",
        "ads",
        "search",
        "pagination",
        "pager",
        "toc",
        "table-of-contents",
    }

    # Consolidating checking for boilerplate and noisy classes
    # Returns true if the tag is boilerplate or noisy
    def is_boilerplate_or_noisy_class(tag):
        try:
            if tag.name in {"nav", "header", "footer", "aside"}:
                return True
            role = str(tag.attrs.get("role", "")).lower()
            if role in {"navigation", "banner", "contentinfo", "complementary", "search"}:
                return True
            # merge id and class into one string and return true if any of them are in noise_classes
            ident = " ".join([
                str(tag.attrs.get("id", "")),
                " ".join(tag.attrs.get("class", []) if isinstance(tag.attrs.get("class", []), list) else [str(tag.attrs.get("class", ""))]),
            ]).lower()

            words = re.split(r"[\s\-_]+", ident)

            return any(word in noise_classes for word in words)
        except Exception:
            return False

    # Remove boilerplate elems that aren't the main content
    for tag in soup.find_all(True):
        if is_boilerplate_or_noisy_class(tag):
            tag.decompose()

    # Focus on anythign under html main header
    main = (
        soup.find("main")
        or soup.find(attrs={"role": "main"})
        or soup.find("article")
    )

    # if main doesn't exist, pick the largest container
    if main is None:
        candidates = soup.find_all(["article", "main", "section", "div"], limit=5000)
        best_candidate = None
        best_length = 0
        for c in candidates:
            if is_boilerplate_or_noisy_class(c): #ignore boilerplate
                continue
            txt = c.get_text(" ", strip=True)
            if len(txt) > best_length:
                best_candidate = c
                best_length = len(txt)
        main = best_candidate

    root = main if main is not None else soup # use the whole page if main doesn't exist even after the second check

    text = h2t.handle(str(root))

    # Normalize whitespace and remove common boilerplate lines
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    for line in text.split("\n"):
        s = line.strip()
        if not s:
            lines.append("")
            continue
        # Drop lines that are basically just separators
        if len(s) <= 3 and all(ch in "-_=*•·" for ch in s):
            continue
        # Collapse internal whitespace
        s = re.sub(r"\s+", " ", s)
        lines.append(s)

    cleaned = "\n".join(lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()
    cleaned = deduplicate_lines(cleaned)
    cleaned = structure_metadata(cleaned)

    return cleaned

# Deduplicate text
def deduplicate_lines(text):
    seen = set()
    out = []
    for line in text.split("\n"):
        key = line.strip().lower()
        if key and key not in seen:
            seen.add(key)
            out.append(line)
    return "\n".join(out)

# Heuristic for labeling time attribute and location attribute (for things like speaker pages or event info pages)
def structure_metadata(text):
    # time
    text = re.sub(
        r"(\d{1,2}:\d{2}\s*[–-]\s*\d{1,2}:\d{2}\s*(am|pm)?)",
        r"Time: \1",
        text,
        flags=re.IGNORECASE
    )
    #lo
    text = re.sub(
        r"(\d+\s+\w+\s+Hall[^\n]*)",
        r"Location: \1",
        text
    )

    return text


for filename in os.listdir("html"):
    if not filename.endswith(".html"):
        continue

    with open(f"html/{filename}") as f:
        html = f.read()

    text = clean_html_for_rag(html)

    out_name = filename[:-5]  # strip .html
    with open(f"cleaned_text/{out_name}.txt", "w") as f:
        f.write(text)

print(f"Done. Converted {len(os.listdir('html'))} files.")